In [ ]:
# Cell 0: synchronize the repository environment before running the workbench.
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Find the workspace root without confusing member pyproject files for it."""
    for candidate in (start, *start.parents):
        if (
            (candidate / "uv.lock").exists()
            and (candidate / "implementations").is_dir()
            and (candidate / "aieng-forecasting").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not find the agentic-forecasting repository root.")


SETUP_ROOT = find_repo_root(Path.cwd().resolve())
print("Running: uv sync --all-extras --dev --all-packages")
setup_result = subprocess.run(
    ["uv", "sync", "--all-extras", "--dev", "--all-packages"],
    cwd=SETUP_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(setup_result.stdout)
if setup_result.returncode != 0:
    print(setup_result.stderr)
    raise RuntimeError("uv sync failed. Install uv or restart the notebook with the repository environment selected.")
print("Environment synchronized successfully.")
print("If packages changed, restart the notebook kernel before continuing.")

# Manufacturing Stress Hybrid Workbench

This is an additive interface for the initial hybrid expansion. It uses the full 15-feature set, the fixed logistic and XGBoost candidates, and the existing ADK/Vector-proxy agent path. The current parameter-sweep and quantitative-only agent workflows remain separate.

The default is a dry run: it builds the cutoff-safe numerical anchor and validates the shadow contract without making an LLM call. Set `RUN_AGENT_CALL = True` only when you explicitly want one agent request.

In [ ]:
import json
import os
import sys
from pathlib import Path

for package_root in (SETUP_ROOT / "aieng-forecasting", SETUP_ROOT / "implementations"):
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))

import pandas as pd
import yaml
from aieng.forecasting.evaluation import BacktestSpec
from dotenv import load_dotenv
from manufacturing_stress_forecasting.data import build_manufacturing_stress_service
from manufacturing_stress_forecasting.hybrid import (
    HybridShadowPredictor,
    build_hybrid_agent_predictor,
    build_numerical_anchor,
    validate_agent_adjustment,
)

REPO_ROOT = SETUP_ROOT
load_dotenv(REPO_ROOT / ".env", override=False)
SPEC_PATH = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "specs" / "manufacturing_stress_smoke.yaml"
print(f"Repository: {REPO_ROOT}")
print(f"FRED API key loaded: {bool(os.getenv('FRED_API_KEY'))}")

## Controls

The fixed target is `manufacturing_stress`: a trailing three-month IPMAN decline of at least 2%, forecast three months ahead. The two statistical candidates use all 15 active features.

In [ ]:
AS_OF = '2017-01-01'
REFRESH_INPUT_DATA = False
RUN_AGENT_CALL = False
MAX_AGENT_ADJUSTMENT = 0.03

print({
    'as_of': AS_OF,
    'refresh_input_data': REFRESH_INPUT_DATA,
    'run_agent_call': RUN_AGENT_CALL,
    'max_agent_adjustment': MAX_AGENT_ADJUSTMENT,
    'feature_set': 'full_15',
    'candidates': ['logistic_c_0_001', 'xgb_50_depth2_lr0_03'],
})

## Load input data

The next cell builds the cutoff-aware data service from the repository cache. With `REFRESH_INPUT_DATA = False`, it uses cached FRED and Yahoo Finance files. Set it to `True` only when you intentionally want to refresh the source data; a valid `FRED_API_KEY` is required for FRED cache misses or refreshes.

In [ ]:
with SPEC_PATH.open() as file:
    spec = BacktestSpec.model_validate(yaml.safe_load(file))

service = build_manufacturing_stress_service(
    cache_dir=REPO_ROOT / 'data' / 'fred',
    yahoo_cache_dir=REPO_ROOT / 'data' / 'yfinance',
    refresh=REFRESH_INPUT_DATA,
)
context = service.context(as_of=pd.Timestamp(AS_OF).to_pydatetime())
anchor = build_numerical_anchor(spec.task, context)
validation = validate_agent_adjustment(anchor.probability, anchor.probability, max_adjustment=MAX_AGENT_ADJUSTMENT)

print({
    'target': spec.task.target_series_id,
    'threshold': '-2%',
    'horizon_months': spec.task.horizons[0],
    'logistic_probability': anchor.logistic_probability,
    'xgboost_probability': anchor.xgboost_probability,
    'anchor_probability': anchor.probability,
    'validation_status': validation['validation_status'],
})

## Optional LLM shadow call

This cell uses the existing manufacturing-stress ADK runner and Vector-proxy configuration. The official probability remains the numerical anchor; the agent result is recorded as shadow metadata and its adjustment is bounded in Python.

In [ ]:
if RUN_AGENT_CALL:
    shadow_predictor = HybridShadowPredictor(build_hybrid_agent_predictor())
    prediction = shadow_predictor.predict(spec.task, context)[0]
    print(json.dumps({
        'official_probability': prediction.payload.probability,
        'shadow_probability': prediction.metadata['shadow_probability'],
        'agent_adjustment': prediction.metadata['agent_adjustment'],
        'validation_status': prediction.metadata['validation_status'],
    }, indent=2))
else:
    print('LLM call skipped; dry-run mode is active.')